In [1]:
# ============================================
# IPL Win Probability Predictor
# Day 4 — Feature Engineering
# Phase 5 — Building Model-Ready Features
# ============================================
# GOAL: Calculate current_score, wickets, run rates 
# for all matches at once, fix division-by-zero edge 
# case, and create the final target label (win/loss)

import pandas as pd
import numpy as np

df = pd.read_csv("../data/cleaned_inning2.csv")
print("✅ Loaded cleaned dataset")
print("Shape:", df.shape)
df.head()

✅ Loaded cleaned dataset
Shape: (83933, 26)


,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,extra_runs,total_runs,player_dismissed,dismissal_kind,fielder,target,id,winner,team1,team2
0,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,1,CH Gayle,Mandeep Singh,A Nehra,0,...,0,1,NaN,NaN,NaN,208,1,Sunrisers Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore
1,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,2,Mandeep Singh,CH Gayle,A Nehra,0,...,0,0,NaN,NaN,NaN,208,1,Sunrisers Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore
2,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,3,Mandeep Singh,CH Gayle,A Nehra,0,...,0,0,NaN,NaN,NaN,208,1,Sunrisers Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore
3,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,4,Mandeep Singh,CH Gayle,A Nehra,0,...,0,2,NaN,NaN,NaN,208,1,Sunrisers Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore
4,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,1,5,Mandeep Singh,CH Gayle,A Nehra,0,...,0,4,NaN,NaN,NaN,208,1,Sunrisers Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore


In [2]:
df.columns

Index(['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball',
       'batsman', 'non_striker', 'bowler', 'is_super_over', 'wide_runs',
       'bye_runs', 'legbye_runs', 'noball_runs', 'penalty_runs',
       'batsman_runs', 'extra_runs', 'total_runs', 'player_dismissed',
       'dismissal_kind', 'fielder', 'target', 'id', 'winner', 'team1',
       'team2'],
      dtype='object')

In [3]:
# Sort to make sure balls are in correct order within each match
df=df.sort_values(['match_id','over','ball']).reset_index(drop=True)

# Cumulative score WITHIN each match separately
df['current_score']=df.groupby('match_id')['total_runs'].cumsum()

df[['match_id', 'over', 'ball', 'total_runs', 'current_score']].head(10)

,match_id,over,ball,total_runs,current_score
0,1,1,1,1,1
1,1,1,2,0,1
2,1,1,3,0,1
3,1,1,4,2,3
4,1,1,5,4,7
5,1,1,6,4,11
6,1,2,1,0,11
7,1,2,2,0,11
8,1,2,3,1,12
9,1,2,4,0,12


In [4]:
# Mark each ball as wicket (1) or not (0)
df['is_wicket']=df['player_dismissed'].notna().astype(int)

# Cumulative wickets WITHIN each match
df['wicket_fallen']=df.groupby('match_id')['is_wicket'].cumsum()

df[['match_id', 'over', 'ball', 'is_wicket', 'wicket_fallen']].head(10)

,match_id,over,ball,is_wicket,wicket_fallen
0,1,1,1,0,0
1,1,1,2,0,0
2,1,1,3,0,0
3,1,1,4,0,0
4,1,1,5,0,0
5,1,1,6,0,0
6,1,2,1,0,0
7,1,2,2,0,0
8,1,2,3,0,0
9,1,2,4,0,0


In [5]:
# Balls bowled so far in this innings
df['balls_bowled']=(df['over']-1)*6+df['ball']

# Balls remaining out of 120 (20 overs x 6 balls)
df['balls_remaining']=120-df['balls_bowled']

df[['match_id', 'over', 'ball', 'balls_bowled', 'balls_remaining']].head(10)

,match_id,over,ball,balls_bowled,balls_remaining
0,1,1,1,1,119
1,1,1,2,2,118
2,1,1,3,3,117
3,1,1,4,4,116
4,1,1,5,5,115
5,1,1,6,6,114
6,1,2,1,7,113
7,1,2,2,8,112
8,1,2,3,9,111
9,1,2,4,10,110


In [6]:
# Fix: cap balls_bowled at 120 maximum (handles extra balls from wides/no-balls)
df['balls_bowled'] = df['balls_bowled'].clip(upper=120)
df['balls_remaining'] = 120 - df['balls_bowled']

print("Min balls_remaining after fix:", df['balls_remaining'].min())
print("Max balls_remaining after fix:", df['balls_remaining'].max())
print("Any negative values left?", (df['balls_remaining'] < 0).sum())

Min balls_remaining after fix: 0
Max balls_remaining after fix: 119
Any negative values left? 0


In [7]:
df['runs_remaining']=df['target']-df['current_score']

# Runs remaining shouldn't go negative even if team overshoots target
df['runs_remaining']=df['runs_remaining'].clip(lower=0)

df[['match_id', 'current_score', 'target', 'runs_remaining']].tail(10)

,match_id,current_score,target,runs_remaining
83923,11415,140,153,13
83924,11415,140,153,13
83925,11415,142,153,11
83926,11415,150,153,3
83927,11415,151,153,2
83928,11415,152,153,1
83929,11415,154,153,0
83930,11415,155,153,0
83931,11415,157,153,0
83932,11415,157,153,0


In [8]:
# Create a "safe" version of balls_remaining just for rate calculations

#current run rate
df['current_run_rate']=np.where(df['balls_bowled']>0,df['current_score']*6/df['balls_bowled'],0)

#required run rate
df['required_run_rate']=np.where(df['balls_remaining']>0,df['runs_remaining']*6/df['balls_remaining'],0)

print("Any infinite values in required_run_rate?", np.isinf(df['required_run_rate']).sum())
print("Any infinite values in current_run_rate?", np.isinf(df['current_run_rate']).sum())

Any infinite values in required_run_rate? 0
Any infinite values in current_run_rate? 0


In [9]:
# 1 if batting team won, 0 if batting team lost
df['batting_team_won']=(df['batting_team']==df['winner']).astype(int)

df[['match_id', 'batting_team', 'winner', 'batting_team_won']].drop_duplicates('match_id').head(10)

,match_id,batting_team,winner,batting_team_won
0,1,Royal Challengers Bangalore,Sunrisers Hyderabad,0
123,2,Rising Pune Supergiants,Rising Pune Supergiant,0
245,3,Kolkata Knight Riders,Kolkata Knight Riders,1
341,4,Kings XI Punjab,Kings XI Punjab,1
463,5,Delhi Daredevils,Royal Challengers Bangalore,0
587,6,Sunrisers Hyderabad,Sunrisers Hyderabad,1
683,7,Mumbai Indians,Mumbai Indians,1
809,8,Kings XI Punjab,Kings XI Punjab,1
898,9,Rising Pune Supergiants,Delhi Daredevils,0
999,10,Mumbai Indians,Mumbai Indians,1


In [10]:
# Remove rows where the chase was already decided (leakage prevention)
df_filtered = df[df['runs_remaining'] > 0].copy()

print("Before removing trivial-outcome rows:", len(df))
print("After removing trivial-outcome rows:", len(df_filtered))
print("Rows removed:", len(df) - len(df_filtered))

Before removing trivial-outcome rows: 83933
After removing trivial-outcome rows: 83480
Rows removed: 453


In [11]:
final_features = [
    'match_id', 'batting_team', 'bowling_team', 
    'current_score', 'wicket_fallen', 'balls_remaining',
    'runs_remaining', 'current_run_rate', 'required_run_rate',
    'batting_team_won'
]

model_df = df_filtered[final_features].copy()
print("Final model dataset shape:", model_df.shape)
model_df.head()

Final model dataset shape: (83480, 10)


,match_id,batting_team,bowling_team,current_score,wicket_fallen,balls_remaining,runs_remaining,current_run_rate,required_run_rate,batting_team_won
0,1,Royal Challengers Bangalore,Sunrisers Hyderabad,1,0,119,207,6.0,10.436975,0
1,1,Royal Challengers Bangalore,Sunrisers Hyderabad,1,0,118,207,3.0,10.525424,0
2,1,Royal Challengers Bangalore,Sunrisers Hyderabad,1,0,117,207,2.0,10.615385,0
3,1,Royal Challengers Bangalore,Sunrisers Hyderabad,3,0,116,205,4.5,10.603448,0
4,1,Royal Challengers Bangalore,Sunrisers Hyderabad,7,0,115,201,8.4,10.486957,0


In [12]:
model_df.to_csv("../data/model_ready_data.csv", index=False)
print("✅ Saved model-ready dataset")
print("Final shape:", model_df.shape)
print("\nColumns:", model_df.columns.tolist())

✅ Saved model-ready dataset
Final shape: (83480, 10)

Columns: ['match_id', 'batting_team', 'bowling_team', 'current_score', 'wicket_fallen', 'balls_remaining', 'runs_remaining', 'current_run_rate', 'required_run_rate', 'batting_team_won']
